In [30]:
# Cell 1
import os
import pandas as pd
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader
from transformers import BertTokenizerFast, BertForSequenceClassification
from torch.optim import AdamW
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, f1_score
from tqdm import tqdm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)
print("Working dir ../data contains:", os.listdir("../data"))


Using device: cuda
Working dir ../data contains: ['gitattributes', 'label_names.npy', 'processed_text.csv', 'sample_submission.csv', 'test.csv', 'test_labels.csv', 'tfidf.pkl', 'train.csv', 'X_test_tfidf.npz', 'X_train_tfidf.npz', 'y_test.npy', 'y_train.npy']


In [31]:
# Cell 2
# Load processed text
df = pd.read_csv("../data/processed_text.csv", keep_default_na=False)

# Use column 11 (processed_text)
texts = df.iloc[:, 11].astype(str).tolist()  # ensure strings

# Load saved labels
y_train = np.load("../data/y_train.npy").astype(np.float32)
y_test  = np.load("../data/y_test.npy").astype(np.float32)
label_names = ["toxic", "severe_toxic", "obscene", "threat", "insult", "identity_hate"]

print("Number of texts:", len(texts))
print("y_train shape:", y_train.shape)
print("y_test shape:", y_test.shape)

# Quick check
print("Positive counts per label (train):", y_train.sum(axis=0))


Number of texts: 159571
y_train shape: (127656, 6)
y_test shape: (31915, 6)
Positive counts per label (train): [12238.  1274.  6734.   404.  6263.  1111.]


In [32]:
# Cell 3
from sklearn.model_selection import train_test_split

X_train_texts, X_val_texts, y_train_labels, y_val_labels = train_test_split(
    texts[:len(y_train)],
    y_train,
    test_size=0.1,
    random_state=42
)
 
print("Training samples:", len(X_train_texts))
print("Validation samples:", len(X_val_texts))

# Quick check of positive counts
print("Positive counts per label in training set:", y_train_labels.sum(axis=0))


Training samples: 114890
Validation samples: 12766
Positive counts per label in training set: [10976.  1131.  6034.   371.  5583.   995.]


In [33]:
import numpy as np

y = np.load("../data/y_train.npy")
print("y_train shape:", y.shape)
print("Number of positive labels per class:", y.sum(axis=0))
print("Total positive labels:", y.sum())




y_train shape: (127656, 6)
Number of positive labels per class: [12238  1274  6734   404  6263  1111]
Total positive labels: 28024


In [34]:
# Check number of positive labels
print("Training labels positive counts per column:", y_train_labels.sum(axis=0))
print("Validation labels positive counts per column:", y_val_labels.sum(axis=0))
print("Test labels positive counts per column:", y_test.sum(axis=0))


Training labels positive counts per column: [10976.  1131.  6034.   371.  5583.   995.]
Validation labels positive counts per column: [1262.  143.  700.   33.  680.  116.]
Test labels positive counts per column: [3056.  321. 1715.   74. 1614.  294.]


In [35]:
# Cell 4
from transformers import BertTokenizerFast

tokenizer = BertTokenizerFast.from_pretrained("bert-base-uncased")

def tokenize_texts(texts, max_len=128):
    return tokenizer(
        texts,
        padding=True,
        truncation=True,
        max_length=max_len,
        return_tensors="pt"
    )

train_encodings = tokenize_texts(X_train_texts)
val_encodings   = tokenize_texts(X_val_texts)
test_encodings  = tokenize_texts(texts[len(y_train):])


In [36]:
# Cell 5
from torch.utils.data import Dataset, DataLoader
import torch

class ToxicCommentsDataset(Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = torch.tensor(labels, dtype=torch.float32)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: val[idx] for key, val in self.encodings.items()}
        item["labels"] = self.labels[idx]
        return item

train_dataset = ToxicCommentsDataset(train_encodings, y_train_labels)
val_dataset   = ToxicCommentsDataset(val_encodings, y_val_labels)
test_dataset  = ToxicCommentsDataset(test_encodings, y_test)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset, batch_size=16, shuffle=False)
test_loader  = DataLoader(test_dataset, batch_size=16, shuffle=False)

# Quick batch check
for batch in train_loader:
    y_batch = batch["labels"]
    print("Batch y_batch shape:", y_batch.shape)
    print("Positives per class in this batch:", y_batch.sum(dim=0))
    break

Batch y_batch shape: torch.Size([16, 6])
Positives per class in this batch: tensor([3., 0., 3., 0., 2., 0.])


In [37]:
# Cell 6
num_labels = y_train.shape[1]

model = BertForSequenceClassification.from_pretrained(
    "bert-base-uncased",
    num_labels=num_labels,
    problem_type="multi_label_classification"
).to(device)

optimizer = AdamW(model.parameters(), lr=2e-5)


Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [21]:
#DIAGNOSTIC
# get one batch from your DataLoader
for batch in train_loader:
    y_batch = batch["labels"]
    y_sum = y_batch.sum(dim=0)

    print("Batch y_batch shape:", y_batch.shape)
    print("Positives per class in this batch:", y_sum)
    break

# ---- DIAGNOSTIC: inspect logits on a single batch ----

batch = next(iter(train_loader))

input_ids = batch["input_ids"].to(device)
attention_mask = batch["attention_mask"].to(device)
labels = batch["labels"].to(device)

model.eval()
with torch.no_grad():
    logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
    probs  = torch.sigmoid(logits)

print("\n=== LOGIT DIAGNOSTIC ===")
print("Logits (first sample):", logits[0])
print("Probs  (first sample):", probs[0])
print("Logit range: min =", logits.min().item(), "max =", logits.max().item())



Batch y_batch shape: torch.Size([16, 6])
Positives per class in this batch: tensor([5., 1., 1., 0., 1., 0.])

=== LOGIT DIAGNOSTIC ===
Logits (first sample): tensor([-0.3834,  0.1489, -0.0271, -0.5913,  0.1074, -0.0029], device='cuda:0')
Probs  (first sample): tensor([0.4053, 0.5372, 0.4932, 0.3563, 0.5268, 0.4993], device='cuda:0')
Logit range: min = -0.5952757000923157 max = 0.23864540457725525


In [22]:
#DIAGNOSTIC
from sklearn.metrics import f1_score
import numpy as np

# gather val probs and labels
all_probs = []
all_labels = []

model.eval()
with torch.no_grad():
    for batch in val_loader:
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].cpu().numpy()
        logits = model(input_ids=input_ids, attention_mask=attention_mask).logits
        probs  = torch.sigmoid(logits).cpu().numpy()

        all_probs.append(probs)
        all_labels.append(labels)

all_probs = np.vstack(all_probs)
all_labels = np.vstack(all_labels)

# try thresholds from 0.1 to 0.5
for th in [0.1, 0.2, 0.3, 0.4, 0.5]:
    preds = (all_probs > th).astype(int)
    macro = f1_score(all_labels, preds, average="macro", zero_division=0)
    print(f"Threshold {th}: Macro F1 = {macro:.4f}")


Threshold 0.1: Macro F1 = 0.0717
Threshold 0.2: Macro F1 = 0.0717
Threshold 0.3: Macro F1 = 0.0717
Threshold 0.4: Macro F1 = 0.0711
Threshold 0.5: Macro F1 = 0.0353


In [ ]:
#DIAGNOSTIC
import pandas as pd

df = pd.read_csv("../data/processed_text.csv", header=None, names=["text"])

df["length"] = df["text"].fillna("").apply(lambda x: len(x.split()))

print(df["length"].describe())
print("Number of empty rows:", (df["length"] == 0).sum())
print("Samples of empty entries:")
print(df[df["length"] == 0].head(5))


C:\Users\Yasna\AppData\Local\Temp\ipykernel_3968\912623866.py:4: DtypeWarning: Columns (2,3,4,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed_text.csv", header=None, names=["text"])


count    159572.000000
mean         29.928772
std          47.282953
min           0.000000
25%           7.000000
50%          16.000000
75%          33.000000
max        1250.000000
Name: length, dtype: float64
Number of empty rows: 170
Samples of empty entries:
                                                                                                                    text  \
006ca45465868e64 86.29.244.57|86.29.244.57]] 04:21, 14 May 2007 0 0 0 0 0 0 0 0 862924457862924457 0421 14 may 2007  NaN   
01ad9cd4eb4a53c3 Seems we both have some.                       0 0 0 0 0 0 0 0 seems                                NaN   
05ac7a7a83e4c63a No, it doesn´t.80.228.65.162                   0 0 0 0 0 0 0 0 doesnt8022865162                     NaN   
0612e01f974ff6fa talk:212.121.210.45|talk]]) 11:48, 28          0 0 0 0 0 0 0 0 talk21212121045talk 1148 28          NaN   
067638a445ccd93b Here, here and here.                           0 0 0 0 0 0 0 0 NaN                                

In [26]:
#DIAGNOSTIC
df = pd.read_csv("../data/processed_text.csv", header=None)

print(df.shape)
print(df.head(10))


(159572, 12)
                 0                                                  1      2   \
0                id                                       comment_text  toxic   
1  0000997932d777bf  Explanation\nWhy the edits made under my usern...      0   
2  000103f0d9cfb60f  D'aww! He matches this background colour I'm s...      0   
3  000113f07ec002fd  Hey man, I'm really not trying to edit war. It...      0   
4  0001b41b1c6bb37e  "\nMore\nI can't make any real suggestions on ...      0   
5  0001d958c54c6e35  You, sir, are my hero. Any chance you remember...      0   
6  00025465d4725e87  "\n\nCongratulations from me as well, use the ...      0   
7  0002bcb3da6cb337       COCKSUCKER BEFORE YOU PISS AROUND ON MY WORK      1   
8  00031b1e95af7921  Your vandalism to the Matt Shirvington article...      0   
9  00037261f536c51d  Sorry if the word 'nonsense' was offensive to ...      0   

             3        4       5       6              7       8            9   \
0  severe_toxic

C:\Users\Yasna\AppData\Local\Temp\ipykernel_3968\371549170.py:2: DtypeWarning: Columns (2,3,4,5,6,7,8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv("../data/processed_text.csv", header=None)


In [28]:
#DIAGNOSTIC
import pandas as pd

df = pd.read_csv("../data/processed_text.csv", header=None, low_memory=False)
processed_text = df[11].astype(str)
processed_text.head(20)



0                                        processed_text
1     explanation edit username hardcore metallica f...
2     daww match background colour m seemingly stuck...
3     hey man m try edit war guy constantly remove r...
4     not real suggestion improvement wonder section...
5                       sir hero chance remember page s
6                          congratulation use tool talk
7                                  cocksucker piss work
8     vandalism matt shirvington article revert not ban
9     sorry word nonsense offensive m intend write a...
10                 alignment subject contrary dulithgow
11    fair use rationale imagewonjujpg thank upload ...
12                    bbq man let discuss itmaybe phone
13    hey talk exclusive group wp talibanswho good d...
14    start throw accusation warning let review edit...
15    oh girl start argument stick nose not belong b...
16    juelz santanas age juelz santana year old come...
17                 bye not look come think comme

In [38]:
# Cell 7
from torch.nn import BCEWithLogitsLoss

# Use small subset for quick debug with positive labels
subset_size = 10000
train_subset_indices = np.random.choice(len(train_dataset), subset_size, replace=False)
small_train_dataset = torch.utils.data.Subset(train_dataset, train_subset_indices)
small_train_loader = DataLoader(small_train_dataset, batch_size=8, shuffle=True)

loss_fn = BCEWithLogitsLoss()
epochs = 3

model.train()
for epoch in range(epochs):
    total_loss = 0
    for batch in tqdm(small_train_loader, desc=f"Training epoch {epoch+1}"):
        optimizer.zero_grad()
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        loss = loss_fn(logits, labels)
        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(f"Epoch {epoch+1} average loss: {total_loss/len(small_train_loader):.4f}")


Training epoch 1: 100%|██████████| 1250/1250 [06:16<00:00,  3.32it/s]


Epoch 1 average loss: 0.1488


Training epoch 2: 100%|██████████| 1250/1250 [05:59<00:00,  3.48it/s]


Epoch 2 average loss: 0.1361


Training epoch 3: 100%|██████████| 1250/1250 [04:50<00:00,  4.31it/s]

Epoch 3 average loss: 0.1359


In [40]:
# Cell 8
model.eval()

all_preds = []
all_labels = []

with torch.no_grad():
    for batch in tqdm(small_train_loader, desc="Evaluating subset"):
        input_ids = batch["input_ids"].to(device)
        attention_mask = batch["attention_mask"].to(device)
        labels = batch["labels"].to(device)

        outputs = model(input_ids, attention_mask=attention_mask)
        logits = outputs.logits

        preds = torch.sigmoid(logits)
        preds = (preds > 0.2).int()

        all_preds.append(preds.cpu())
        all_labels.append(labels.cpu())

y_pred = torch.vstack(all_preds).numpy()
y_true = torch.vstack(all_labels).numpy()

macro_f1 = f1_score(y_true, y_pred, average="macro")
print(f"Macro F1-score (small subset): {macro_f1:.4f}\n")

print("Classification Report (small subset):\n")
print(classification_report(y_true, y_pred, digits=4, target_names=label_names))



Evaluating subset: 100%|██████████| 1250/1250 [01:11<00:00, 17.37it/s]
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 due to no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])
c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in samples with no predicted labels. Use `zero_division` parameter to control this b

Macro F1-score (small subset): 0.0000

Classification Report (small subset):

               precision    recall  f1-score   support

        toxic     0.0000    0.0000    0.0000       908
 severe_toxic     0.0000    0.0000    0.0000       110
      obscene     0.0000    0.0000    0.0000       494
       threat     0.0000    0.0000    0.0000        33
       insult     0.0000    0.0000    0.0000       449
identity_hate     0.0000    0.0000    0.0000        73

    micro avg     0.0000    0.0000    0.0000      2067
    macro avg     0.0000    0.0000    0.0000      2067
 weighted avg     0.0000    0.0000    0.0000      2067
  samples avg     0.0000    0.0000    0.0000      2067



c:\Users\Yasna\Desktop\CMD Project 1\venv\Lib\site-packages\sklearn\metrics\_classification.py:1731: UndefinedMetricWarning: F-score is ill-defined and being set to 0.0 in samples with no true nor predicted labels. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", result.shape[0])


In [41]:
#DIAGNOSTIC
import torch

model.eval()

# -------------------------------
# 1. Grab 1 example PER CLASS that is positive
# -------------------------------
pos_indices = {}

for i, row in enumerate(y_train_labels):
    for label_idx in range(6):
        if row[label_idx] == 1 and label_idx not in pos_indices:
            pos_indices[label_idx] = i
    if len(pos_indices) == 6:
        break

print("Positive example indices found:", pos_indices)

# sanity check: if not all found -> that's the problem
if len(pos_indices) < 6:
    print("ERROR: Not all classes had positive examples available.")
else:
    # Build a batch of ONLY positive examples
    texts_batch = [X_train_texts[pos_indices[i]] for i in range(6)]
    labels_batch = torch.tensor([[1 if i == j else 0 for j in range(6)] for i in range(6)], dtype=torch.float32)

    print("\nLabels batch:\n", labels_batch)

    # tokenization
    batch = tokenizer(texts_batch, padding=True, truncation=True, return_tensors="pt", max_length=128)
    input_ids = batch["input_ids"].to(device)
    attention_mask = batch["attention_mask"].to(device)
    labels_batch = labels_batch.to(device)

    model.train()  # needed for backward

    # -------------------------------
    # 2. Forward pass BEFORE training
    # -------------------------------
    logits = model(input_ids, attention_mask=attention_mask).logits
    probs_before = torch.sigmoid(logits).detach().cpu()

    print("\n=== BEFORE TRAINING ===")
    print("Logits:\n", logits)
    print("Probs:\n", probs_before)

    # -------------------------------
    # 3. Backward pass (one step)
    # -------------------------------
    optimizer.zero_grad()
    loss = torch.nn.BCEWithLogitsLoss()(logits, labels_batch)
    loss.backward()

    # capture gradients from classifier head
    grads = model.classifier.weight.grad.clone().cpu()

    print("\nClassifier Head Gradients (sum of abs):", grads.abs().sum().item())

    # do NOT do optimizer.step(); we only need to test gradient flow


Positive example indices found: {0: 9, 2: 31, 4: 31, 3: 116, 1: 124, 5: 180}

Labels batch:
 tensor([[1., 0., 0., 0., 0., 0.],
        [0., 1., 0., 0., 0., 0.],
        [0., 0., 1., 0., 0., 0.],
        [0., 0., 0., 1., 0., 0.],
        [0., 0., 0., 0., 1., 0.],
        [0., 0., 0., 0., 0., 1.]])

=== BEFORE TRAINING ===
Logits:
 tensor([[-2.3707, -4.6544, -3.5436, -5.7988, -3.4023, -4.6088],
        [-2.4547, -4.7797, -3.2498, -5.7353, -3.2083, -4.9107],
        [-2.4753, -4.8019, -3.3769, -5.7622, -3.4585, -4.7330],
        [-2.2700, -4.6774, -3.6034, -5.4159, -3.2610, -4.9018],
        [-2.4786, -4.5920, -3.0091, -5.5464, -3.3676, -4.6557],
        [-2.5151, -4.7484, -3.2318, -5.9024, -3.1465, -4.8519]],
       device='cuda:0', grad_fn=<AddmmBackward0>)
Probs:
 tensor([[0.0854, 0.0094, 0.0281, 0.0030, 0.0322, 0.0099],
        [0.0791, 0.0083, 0.0373, 0.0032, 0.0389, 0.0073],
        [0.0776, 0.0081, 0.0330, 0.0031, 0.0305, 0.0087],
        [0.0936, 0.0092, 0.0265, 0.0044, 0.0369, 0.